![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 40 -- Case Study: Source Extraction (Solution)

The task of this morning's case study, as the contestants of the **Polish AI Olympiad 2025 (stage 2)** received it.
The same notebook with the class `Embedder` written as on the slides: weighted-mean pooling, the special brackets, and texts of up to 300 tokens.

**Source and licence.** Adapted from the notebook `2_etap/ekstrakcja_zrodel/ekstrakcja_zrodel.ipynb` of
[OlimpiadaAI/II-OlimpiadaAI](https://github.com/OlimpiadaAI/II-OlimpiadaAI) by the organisers of the Polish AI
Olympiad, licensed [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). This adaptation is
shared under the same licence. Changes: the statement and the comments are translated from Polish; this cell and
the setup cell below are added; the class `Embedder` is written, with a cell that explains it. The data is SciFact (Wadden et al., 2020).

**How to run it.** In Colab, choose *Runtime > Change runtime type > T4 GPU*, then *Run all*. The setup cell
downloads the two data files; the model downloads from the Hugging Face Hub the first time.


In [ ]:
# Course setup (not part of the task): download the task's two data files from the organisers' repository.
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/OlimpiadaAI/II-OlimpiadaAI/main/2_etap/ekstrakcja_zrodel/"
for name in ["corpus.jsonl", "queries_val.jsonl"]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(BASE + name, name)
print(sorted(f for f in os.listdir() if f.endswith(".jsonl")))

# Source Extraction

<img src="https://live.staticflickr.com/65535/54443002259_4a8e1249dd_b.jpg" alt="Embedded Photo" width="500">

*Image generated with ChatGPT.*

## Introduction

Language models can be prone to saying things that are false or half true, and to making up facts without giving
sources. More and more often, systems are used that do not answer a question directly: they first search a database,
for example a collection of documents, and only then generate an answer from the documents that fit best. Such an
answer is more likely to be grounded in reality, and a person can check it -- as long as the right sources were found.

Of course, there can be very many sources, so the search methods must be efficient: processing everything "at once"
directly with a language model is out of the question! In this task you will focus on finding the best sources for a
given sentence, using **embeddings** (vectors that represent the texts).

Imagine you are an AI engineer at a company building a tool that checks scientific facts. Your job is to build a
module that quickly and reliably finds trustworthy scientific publications that support or refute specific claims.
With your solution, scientists, journalists and decision makers will be able to check information against solid
scientific evidence, which matters especially in an age of disinformation.


## Task

Your task is to build a system that produces high-quality vector representations (embeddings) of both the queries and
the source documents, so that the right sources can be matched precisely to the queries.

Given the **queries** (the claims we look for sources for) and the **corpus** (the base of documents, the sources
considered), you must implement functions that give the queries and the sources vectors of real numbers of
dimension $768$. These vectors will be used to find the sources for every query by the evaluation function we
provide, which, for a given query, picks from the documents the $k=10$ nearest neighbours ($k$-Nearest Neighbours).

In your solution you may use the provided model based on the GPT2 architecture, which was specially fine-tuned to be
helpful in obtaining good-quality embeddings.

While working on your solution you can test how well it works on the validation set, which lets you judge the quality
of the embeddings for the task of finding the right source documents.

### Data

The data available to you in this task:

- A set of queries, for which the right sources must be found
- A corpus of documents, holding the scientific publications that can be sources for the queries
- Which document matches which query, for the validation set

Your solution will be evaluated on the *SciFact* benchmark. It is used to evaluate systems that search for and verify
facts in a scientific context. It is made of claims (queries) based on real scientific publications, and the document
base (corpus) is publications from the natural and medical sciences. For every claim there is at least one
publication that supports or refutes it. We provide the code that loads the data, so the data is described here for
information only.


**The file `corpus.jsonl`** holds the unique ids, the titles and the abstracts of scientific papers.

An example of one document:
```
{
    "text_id": 13734012,
    "title": "Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey",
    "text": "OBJECTIVES To carry out a further survey (...) CONCLUSIONS This study corroborates previous studies and suggests a high prevalence of infection with abnormal PrP, indicating vCJD carrier status in the population compared with the 177 vCJD cases to date. These findings have important implications for the management of blood and blood products and for the handling of surgical instruments."
}
```

**The file `queries_val.jsonl`** holds the texts of the claims and the id of the matching source text. The test set,
on which your solution will finally be evaluated, **will not contain** the ids of the matching source texts.

An example of one query:
```
{
    "query": "1 in 5 million in UK have abnormal PrP positivity.",
    "matching_text_id": 13734012
}
```

### Evaluation Criterion
The methods (functions) you implement, `Embedder.encode_queries` and `Embedder.encode_corpus`, will be used to turn
the queries $q \in Q$ and the documents $d \in C$ into vectors. Below we use $q$ and $d$ both for the texts and for
their embeddings.

Suppose the query $q\in Q$ has the gold document $d\in C$.
The evaluation code sorts all documents by their distance from $q$, giving the documents $K_1, K_2, ..., K_n$, with
$K_1$ the closest. We write $I$ for the index of the gold document $d$ in this sequence. That is, $I - 1$ is the
number of documents whose distance from $q$ is smaller than the distance between $q$ and $d$.

The distance between vectors is computed with cosine similarity, which for vectors $v, w \in \mathbb{R}^n$ is defined
as $\frac{v^Tw}{||v|| \cdot ||w||}$, where $||v||$ is the length of the vector $v$.

The score for the query $q$ is

$$\text{nDCG@10}(q) = \begin{cases}
\frac{1}{\log_2(I + 1)} & \text{if $I \leq 10$} \\
0 & \text{otherwise.}
\end{cases}$$

So the closer the gold document is placed to the query compared with the other documents, the higher the score -- if
10 "wrong" documents are closer to the query, the score for this example is 0.

In the end, your solution is graded with the **nDCG@10** metric, computed as the mean of this metric over all the
queries $(q \in Q )$.

- If the **nDCG@10** score is **below 0.2**, you get **0 points**.
- If the score **exceeds 0.5**, you get the **maximum number of points**, which is **100**.

Points for values between these thresholds are given proportionally.


## Limits

- Your solution will be tested on the Contest Platform without internet access, in an environment with a GPU.
- The evaluation of your final solution on the Contest Platform may not take longer than 10 minutes with a GPU.
- The embedding of every query and text must have dimension 768.
- The list of allowed libraries: `torch`, `pandas`, `numpy`, `nltk`, `transformers`.

## Submission Files

Submit only this notebook, completed with your solution (see the class `Embedder`).

## Hints

- The GPT2 model is a decoder-type language model. Decoder models work like this: for a given sequence of tokens (for
  example a prefix of the sentence being processed) $t_1, t_2, \dots, t_n$ they compute a hidden vector
  $h_{n+1} \in \mathbb{R}^d$, and then transform it with one of their weight matrices into $p_{n+1} \in \mathbb{R}^m$
  -- a probability distribution over the tokens of the vocabulary.
- Compared with the available running time, there are many documents.

## Evaluation

During grading, the flag `FINAL_EVALUATION_MODE` will be set to `True`.

For this task you can earn between 0 and 100 points. Your points will be computed on the (secret) test set on the
Contest Platform with the formula above, rounded to a whole number. If your solution does not meet the criteria above
or does not run correctly, you get 0 points for the task.

# Starter Code

In this section we set up the environment by importing the libraries and functions we need. The prepared code for
the tokenizer, for loading the data and for the evaluation makes it easier to work with the data and lets you solve
the task.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

FINAL_EVALUATION_MODE = False  # While your solution is graded, we will change this value to True

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

import json
import os
from math import log2

import torch
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Tokenizer:
    def __init__(self, tokenizer_path, length=150):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "right"
        self.length = length

    def __call__(self, batch_text):
        batch_tensor = self.tokenizer(
            batch_text,
            max_length=self.length,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )
        return batch_tensor.to(device)

## Loading the Data
In this part of the task we load the training data.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

def load_corpus(file):
    corpus = {}
    with open(file, encoding="utf8") as f_in:
        for line in f_in:
            line = json.loads(line)
            corpus[line.get("text_id")] = {
                "text": line.get("text"),
                "title": line.get("title"),
            }
    return corpus

def load_queries(file):
    queries = {}
    matching_texts = {}
    with open(file, encoding="utf8") as f_in:
        for query_num, line in enumerate(f_in):
            line = json.loads(line)

            queries[query_num] = line.get("query")
            matching_texts[query_num] = line.get("matching_text_id")
    return queries, matching_texts

corpus = load_corpus("corpus.jsonl")
queries, matching_texts = load_queries("queries_val.jsonl")

print(f"Loaded {len(corpus)} texts and {len(queries)} queries.")

## The Evaluation Code

Code close to the code below will be used to evaluate the solution on the test set.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

def evaluate_retrieval_ndcg(
    golden_matches: dict[int, int],
    results: dict[int, dict[int, float]],
) -> float:
    """
    Computes the nDCG metric for the given search results.

    The function computes the score of your solution from the results for the $top\\_k$ best documents according to your embedder.

    :param golden_matches: A dictionary of the gold matches, where the key is the id of a query and the value is the id of the right document.
    :param results: A dictionary of search results, where the key is the id of a query and the value is a dictionary of document ids and their similarities to that query.
    :return: The value of the nDCG metric."""

    for query_id, v in results.items():
        results[query_id] = {k: v for k, v in sorted(v.items(), key=lambda item: -item[1])}

    ndcg_sum = 0
    for query_id, v in results.items():
        golden_document = golden_matches[query_id]
        for i, document_id in enumerate(v.keys()):
            if golden_document == document_id:
                ndcg_sum += 1 / log2(i + 2)

    ndcg = round(ndcg_sum / len(results), 5)
    return ndcg


def compute_score(ndcg: float) -> float:
    """
    Computes the points from the value of the nDCG metric.
    """
    lower_bound = 0.2
    upper_bound = 0.5

    if ndcg <= lower_bound:
        return 0
    elif lower_bound < ndcg < upper_bound:
        return int(round(100 * (ndcg - lower_bound) / (upper_bound - lower_bound)))
    else:
        return 100

### Search
Below is the code that picks, for a given query, the $top\_k$ best documents from the corpus.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

def cos_sim(a: torch.Tensor, b: torch.Tensor):
    """
    Computes the cosine similarity cos_sim(a[i], b[j]) for all i and j.
    :return: Matrix with res[i][j]  = cos_sim(a[i], b[j])
    """
    a_norm = torch.nn.functional.normalize(a, p=2, dim=1)
    b_norm = torch.nn.functional.normalize(b, p=2, dim=1)
    return torch.mm(a_norm, b_norm.transpose(0, 1))

def search_topk_texts(
    embedder,
    corpus: dict[str, dict[str, str]],
    queries: dict[str, str],
    top_k: int = 10,
) -> dict[str, dict[str, float]]:
    results = {}

    # Create embeddings for all queries using model.encode_queries()
    # Runs semantic search against the corpus embeddings
    # Returns a ranked list with the corpus ids
    query_ids = list(queries.keys())
    results = {qid: {} for qid in query_ids}
    queries = [queries[qid] for qid in queries]
    query_embeddings = embedder.encode_queries(queries)

    corpus_ids = sorted(
        corpus,
        key=lambda k: len(corpus[k].get("title", "") + corpus[k].get("text", "")),
        reverse=True,
    )
    corpus = [corpus[cid] for cid in corpus_ids]

    # Encode chunk of corpus
    corpus_embeddings = embedder.encode_corpus(corpus)

    # Compute similarites using cosine-similarity
    cos_scores = cos_sim(query_embeddings, corpus_embeddings)
    cos_scores[torch.isnan(cos_scores)] = -1

    # Get top-k values
    cos_scores_top_k_values, cos_scores_top_k_idx = torch.topk(
        cos_scores,
        min(top_k + 1, len(cos_scores[1])),
        dim=1,
        largest=True,
        sorted=False,
    )
    cos_scores_top_k_values = cos_scores_top_k_values.cpu().tolist()
    cos_scores_top_k_idx = cos_scores_top_k_idx.cpu().tolist()

    for query_itr in range(len(query_embeddings)):
        query_id = query_ids[query_itr]
        for score, corpus_id in zip(cos_scores_top_k_values[query_itr], cos_scores_top_k_idx[query_itr]):
            results[query_id][corpus_ids[corpus_id]] = score

    return results

# Your Solution
Put your solution in this section. Make changes only here!

**How this solution works** (the Day 40 slides, *The Solution, Step by Step*). Nothing is trained: the given
model is used the way its authors trained it, and each choice comes from one fact found while reading the task.

1. **Brackets.** The model's name ends in `specb`: in training it saw every query as `[ ... ]` and every document as
   `{ ... }`. The bracket ids are added after tokenizing, so a bracket never merges with the word next to it.
2. **Length.** The model was trained on texts of 300 tokens; the given default of 150 cuts 95% of the abstracts.
   Each text keeps 298 tokens, plus its two brackets.
3. **Pads on the right.** The given tokenizer fills short rows with pads after the text; the attention mask marks them
   with 0, so the pooling gives them no weight.
4. **Weighted-mean pooling.** The name says `weightedmean`: GPT-2 is a decoder, so token $i$ has read tokens $1$ to $i$,
   and its weight is $w_i = i / \sum_j j$ over the real tokens.

Measured with the evaluation cell below on the 300 validation claims: **nDCG@10 = 0.549, 100 points** (the published
solution with the plain mean at 256 tokens: 0.463, 88 points). The 5,183 abstracts take about 2 minutes on a GPU.

In [ ]:
MODEL = "Muennighoff/SGPT-125M-weightedmean-msmarco-specb-bitfit"


class Embedder:
    # Do not change the signature of the constructor
    def __init__(self):
        self.tokenizer = Tokenizer(MODEL).tokenizer           # the given settings: pads on the right
        self.model = AutoModel.from_pretrained(MODEL).to(device).eval()
        # the ids of the brackets: [ ] for a query (58, 60), { } for a document (90, 92)
        self.query_marks = (self.tokenizer.encode("[")[0], self.tokenizer.encode("]")[0])
        self.doc_marks = (self.tokenizer.encode("{")[0], self.tokenizer.encode("}")[0])

    @staticmethod
    def weighted_mean(hidden, attention_mask):
        # hidden: (B, S, 768), attention_mask: (B, S); returns (B, 768)
        mask = attention_mask.unsqueeze(-1).float()               # (B, S, 1): 1 on a token, 0 on a pad
        pos = torch.arange(1, hidden.shape[1] + 1, device=hidden.device)
        w = pos.view(1, -1, 1).float() * mask                     # weight i on a token, 0 on a pad
        v = (hidden * w).sum(dim=1) / w.sum(dim=1)                # sum of w_i h_i, divided by the sum of w_i
        return torch.nn.functional.normalize(v, dim=1)            # length 1: cosine = dot product

    @torch.no_grad()                                              # nothing is trained
    def embed(self, texts, marks):
        open_id, close_id = marks
        vectors = []
        for i in range(0, len(texts), 32):                        # 32 texts at a time, in the given order
            rows = [[open_id] + self.tokenizer.encode(t)[:298] + [close_id] for t in texts[i:i + 32]]
            batch = self.tokenizer.pad({"input_ids": rows}, return_tensors="pt").to(device)   # pads and mask
            hidden = self.model(**batch).last_hidden_state        # (32, S, 768)
            vectors.append(self.weighted_mean(hidden, batch["attention_mask"]))
        return torch.cat(vectors)                                 # (len(texts), 768), on the device

    def encode_queries(self, queries: list[str]):
        return self.embed(queries, self.query_marks)              # [ claim ]

    def encode_corpus(self, texts: list[dict]):
        docs = [t["title"] + " " + t["text"] for t in texts]      # one text per abstract: its title and its text
        return self.embed(docs, self.doc_marks)                   # { title text }

# Evaluation

Running the cell below shows how many points your solution would get on the validation data. Before submitting, make
sure the whole notebook runs from start to end without errors and without any action from you after choosing
"Run All".

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
    embedder = Embedder()

    with torch.no_grad():
        results = search_topk_texts(embedder, corpus, queries, top_k=10)

    # Compute the nDCG
    ndcg = evaluate_retrieval_ndcg(matching_texts, results)

    # Compute the final points from the nDCG
    points = compute_score(ndcg)

    print(f"\nNumber of queries: {len(queries)}")
    print(f"Number of texts: {len(corpus)}")
    print(f"nDCG: {ndcg:.3f}")
    print(f"Points: {points}")


During grading, the model will be saved as `your_model.pkl` and evaluated on the test set.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

if FINAL_EVALUATION_MODE:
    import cloudpickle

    OUTPUT_PATH = "file_output"
    FUNCTION_FILENAME = "your_model.pkl"
    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, FUNCTION_FILENAME)

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(Embedder, f)

Adapted By **Sattam Altwaim**, from the notebook of the organisers of the Polish AI Olympiad (CC BY-NC-SA 4.0).